# 11 — Robustness checks (issue #35)

Three things turned up while building notebooks 07–10 that could undercut their conclusions. This notebook tests each one before the Project 1 report relies on them:

1. **Coding era.** Our patient graphs are built from raw billing codes. ICD-9 (before October 2015) and ICD-10 (after) codes for the same disease share nothing, so patients from the two eras can barely link. Does that split *explain* the homophily we found?
2. **SNOMED-harmonized graphs.** Map both coding systems to shared SNOMED concepts (notebook 08's fix), rebuild every graph, and re-run the full analysis. Do the conclusions hold once the era split is gone?
3. **Clean Hypertension cohort.** 55 of 515 "new hypertension" positives already have an essential-hypertension code *before* the prediction time, though EHRSHOT's definition says such discharges are excluded. Re-run Task B without those patients.

Official labels and raw codes stay the **primary** analysis, for comparability with Stanford's benchmark and with notebooks 07–10. Everything here is a sensitivity analysis. All computation goes through the same `ehrgraph.netstats.task_analysis` code, and only aggregate results are written.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"
omop_dir = assets / "femr" / "logs" / "omop_dir"

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ehrgraph.data.cohort import patient_code_counts
from ehrgraph.data.loader import open_database
from ehrgraph.data.vocabulary import (
    MappedConditions,
    descendant_concepts,
    load_concept_table,
    standard_concept_map,
)
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.netstats.task_analysis import (
    PRIMARY_VARIANT,
    analyze_task,
    coding_era,
    era_stats,
    load_task_graphs,
)
from ehrgraph.viz import network_figures as nf

TASKS = {"guo_los": "Task A: Long Length of Stay", "new_hypertension": "Task B: Hypertension"}
SNOMED_VARIANT = "snomed_binary_perpatient_tfidf_mutual_k10"
CLEAN_VARIANT = "cond_binary_perpatient_tfidf_mutual_k10_clean_cohort"
results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
nf.apply_style()

db = open_database(str(meds_reader_path))
primary = pd.concat(
    [pd.read_csv(results_dir / f) for f in ("network_stats_task_a.csv", "network_stats_task_b.csv")]
)
primary = primary[primary["seed"].isna()]
rows = []  # everything this notebook writes


def era_rows(task, variant, graphs, era, labels):
    out = []
    for condition in ("safe", "leaky"):
        s = era_stats(graphs[condition], era, labels)
        for metric in (
            "icd9_icd10_cross_edges",
            "icd9_icd10_cross_edge_fraction",
            "era_assortativity",
        ):
            out.append(
                dict(
                    task=task,
                    condition=condition,
                    graph_variant=variant,
                    metric=metric,
                    value=s[metric],
                )
            )
        out.append(
            dict(
                task=task,
                condition=condition,
                graph_variant=variant,
                metric="homophily_era_only",
                value=s["homophily_era_only"],
                ci_low=s["homophily_era_only_lo"],
                ci_high=s["homophily_era_only_hi"],
            )
        )
    return out

## Part 1 — Coding era in the primary (raw-code) graphs

Each node gets an era from its safe diagnosis vector:
- **icd9** or **icd10**: only one coding system appears.
- **mixed**: both appear.
- **none**: no diagnoses before the prediction time.

Then three numbers per graph:
- **Cross-era links:** how many links join an ICD-9-only node to an ICD-10-only node.
- **Era assortativity:** how strongly nodes link within their own era (1 = only ever within era).
- **Era-only homophily:** shuffle outcomes *within* each era, keeping each era's outcome rate, and re-measure homophily. Whatever survives is what the era split alone could produce. If it is close to the real homophily, our "outcome signal" was really calendar time.

In [ ]:
t0 = time.time()
raw = {task: load_task_graphs(db, assets, task, n_null=1) for task in TASKS}
eras = {}
for task, tg in raw.items():
    matrix, codes = tg.vectors["safe"]
    eras[task] = dict(zip(tg.nodes, coding_era(matrix, codes), strict=True))
print(f"raw graphs rebuilt in {time.time() - t0:.0f}s")

summary = []
for task, tg in raw.items():
    d = pd.DataFrame(
        {
            "era": [eras[task][n] for n in tg.nodes],
            "y": [tg.labels[n] for n in tg.nodes],
            "year": [n[1].year for n in tg.nodes],
        }
    )
    for era_name, grp in d.groupby("era"):
        summary.append(
            {
                "task": TASKS[task],
                "era": era_name,
                "nodes": len(grp),
                "outcome rate": round(grp["y"].mean(), 3),
                "prediction years": f"{grp['year'].min()}–{grp['year'].max()}",
            }
        )
    rows += era_rows(task, PRIMARY_VARIANT, tg.graphs, eras[task], tg.labels)
pd.DataFrame(summary).set_index(["task", "era"])

In [ ]:
def era_table(variant):
    era_metrics = {
        "icd9_icd10_cross_edges",
        "icd9_icd10_cross_edge_fraction",
        "era_assortativity",
        "homophily_era_only",
    }
    d = pd.DataFrame(
        [r for r in rows if r["graph_variant"] == variant and r["metric"] in era_metrics]
    )
    t = d.pivot_table(index=["task", "condition"], columns="metric", values="value")
    real = pd.concat([primary, pd.DataFrame(rows)]).query("graph_variant == @variant")
    real = real[real["seed"].isna()].set_index(["task", "condition", "metric"])["value"]
    t["homophily (actual)"] = [
        real.get((task, cond, "class_insensitive_homophily"), np.nan) for task, cond in t.index
    ]
    return t[
        [
            "icd9_icd10_cross_edges",
            "icd9_icd10_cross_edge_fraction",
            "era_assortativity",
            "homophily_era_only",
            "homophily (actual)",
        ]
    ].round(4)


era_table(PRIMARY_VARIANT)

## Part 2 — SNOMED-harmonized patient graphs

Every condition code is mapped to its standard SNOMED concept through the dataset's own vocabulary (`standard_concept_map`). Then the safe, leaky and 10 null graphs are rebuilt for both tasks, and the whole analysis re-runs unchanged. Nothing else differs from notebooks 07 and 10: same censoring, same TF-IDF, same mutual 10-NN, same metrics.

In [ ]:
t0 = time.time()
concepts = load_concept_table(omop_dir / "concept.csv")
code_map = standard_concept_map(
    patient_code_counts(db, list(db), table="condition"),
    concepts,
    omop_dir / "concept_relationship.csv",
)
mapped_db = MappedConditions(db, code_map)
n_concepts = len(set(code_map.values()))
print(
    f"code map ready in {time.time() - t0:.0f}s: {len(code_map):,} codes -> {n_concepts:,} concepts"
)

snomed = {}
for task in TASKS:
    t0 = time.time()
    tg = load_task_graphs(mapped_db, assets, task)
    an = analyze_task(tg, graph_variant=SNOMED_VARIANT)
    snomed[task] = (tg, an)
    rows += an.rows
    rows += era_rows(task, SNOMED_VARIANT, tg.graphs, eras[task], tg.labels)
    print(f"{TASKS[task]}: {len(tg.nodes):,} nodes, analysed in {time.time() - t0:.0f}s")
era_table(SNOMED_VARIANT)

In [ ]:
KEY = {
    "class_insensitive_homophily": "homophily",
    "class_insensitive_homophily_excl_same_patient": "homophily, no same-patient links",
    "leiden_outcome_eta_squared": "outcome explained by community (η²)",
    "dirichlet_energy_clmbr": "Dirichlet energy (CLMBR)",
    "leiden_modularity": "modularity",
    "largest_cc_fraction": "share in largest piece",
    "same_patient_edge_fraction": "same-patient share of links",
    "max_single_code_outcome_gap": "strongest single-code gap",
}


def compare(task, variant_rows, label):
    real = primary[primary["task"] == task].set_index(["condition", "metric"])["value"]
    alt = pd.DataFrame(variant_rows)
    alt = alt[alt["seed"].isna() & (alt["task"] == task)].set_index(["condition", "metric"])[
        "value"
    ]
    out = []
    for metric, name in KEY.items():
        for cond in ("safe", "leaky", "null_model"):
            if (cond, metric) in real.index and (cond, metric) in alt.index:
                out.append(
                    {
                        "measure": name,
                        "graph": cond,
                        "primary (raw codes)": real[(cond, metric)],
                        label: alt[(cond, metric)],
                    }
                )
    return pd.DataFrame(out).set_index(["measure", "graph"]).round(3)


for task in TASKS:
    print(TASKS[task])
    display(compare(task, snomed[task][1].rows, "SNOMED-harmonized"))  # noqa: F821

## Part 3 — Clean Hypertension cohort

EHRSHOT defines Hypertension as a first diagnosis of SNOMED 59621000, *essential hypertension*, or its child concepts within a year of discharge. Discharges where the patient already has the diagnosis are excluded. The shipped vocabulary maps both ICD-9 401.9 and ICD-10 I10 to that concept, yet some patients in the official cohort already carry one of those codes before the prediction time. Here they are counted, then removed, and Task B is re-run.

In [ ]:
# EHRSHOT's definition: SNOMED 59621000 *and its children* (benign, malignant,
# systolic... essential hypertension), so every source code mapping into that subtree.
essential = descendant_concepts(
    ["SNOMED/59621000"], concepts, omop_dir / "concept_relationship.csv"
)
HTN = {code for code, concept in code_map.items() if concept in essential}
tg_b = raw["new_hypertension"]
prior = {}
for node in tg_b.nodes:
    times = [e.time for e in db[node[0]].events if e.code in HTN and e.time < node[1]]
    if times:
        prior[node] = (node[1] - min(times)).total_seconds() / 86400
y = tg_b.labels
pos = [n for n in prior if y[n] == 1]
neg = [n for n in prior if y[n] == 0]
n_pos, n_neg = sum(y.values()), len(y) - sum(y.values())
print(f"codes counted as essential hypertension: {sorted(HTN)}")
print(f"positives with a prior code: {len(pos)} of {n_pos} ({len(pos) / n_pos:.1%})")
print(f"negatives with a prior code: {len(neg)} of {n_neg} ({len(neg) / n_neg:.1%})")
print(
    f"median days from first prior code to prediction time: {np.median(list(prior.values())):.1f}"
)
for metric, value in (
    ("prior_coded_positives", len(pos)),
    ("prior_coded_negatives", len(neg)),
    ("prior_coded_positive_share", len(pos) / n_pos),
    ("prior_coded_negative_share", len(neg) / n_neg),
):
    rows.append(
        dict(
            task="new_hypertension",
            condition="reference",
            graph_variant=PRIMARY_VARIANT,
            metric=metric,
            value=float(value),
        )
    )

In [ ]:
t0 = time.time()
tg_clean = load_task_graphs(db, assets, "new_hypertension", exclude=set(prior))
an_clean = analyze_task(tg_clean, graph_variant=CLEAN_VARIANT)
rows += an_clean.rows
print(f"clean cohort: {len(tg_clean.nodes):,} nodes, prevalence {tg_clean.prevalence:.3f}")
print(f"analysed in {time.time() - t0:.0f}s")
compare("new_hypertension", an_clean.rows, "clean cohort")

## All three checks in one picture

Each panel is one task. Each row is one version of the graphs: primary, SNOMED-harmonized, and (Task B only) the clean cohort. Filled dots are homophily with all links, and open dots are homophily without same-patient links. The green band is the null ensemble's 95% range.

In [ ]:
alt = pd.DataFrame(rows)
alt = alt[alt["seed"].isna()]
both = pd.concat([primary, alt])
VARIANTS = [
    (PRIMARY_VARIANT, "primary (raw codes)"),
    (SNOMED_VARIANT, "SNOMED-harmonized"),
    (CLEAN_VARIANT, "clean cohort"),
]


def v(task, variant, cond, metric, col="value"):
    hit = both[
        (both["task"] == task)
        & (both["graph_variant"] == variant)
        & (both["condition"] == cond)
        & (both["metric"] == metric)
    ]
    return float(hit[col].iloc[0]) if len(hit) else np.nan


fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
for ax, task in zip(axes, TASKS, strict=True):
    present = [
        (var, name)
        for var, name in VARIANTS
        if not np.isnan(v(task, var, "safe", "class_insensitive_homophily"))
    ]
    for row, (var, _name) in enumerate(present):
        y0 = len(present) - 1 - row
        lo = v(task, var, "null_model", "class_insensitive_homophily", "ci_low")
        hi = v(task, var, "null_model", "class_insensitive_homophily", "ci_high")
        ax.hlines(y0, lo, hi, color=nf.COLORS["null"], lw=8, alpha=0.35)
        for dy, cond in ((0.12, "safe"), (-0.12, "leaky")):
            a = v(task, var, cond, "class_insensitive_homophily")
            b = v(task, var, cond, "class_insensitive_homophily_excl_same_patient")
            ax.plot([a, b], [y0 + dy] * 2, color=nf.COLORS[cond], lw=1.2, alpha=0.6)
            ax.plot(a, y0 + dy, "o", color=nf.COLORS[cond], ms=8, label=cond if row == 0 else None)
            ax.plot(b, y0 + dy, "o", mfc="#fcfcfb", mec=nf.COLORS[cond], mew=2, ms=8)
        ax.set_yticks(range(len(present)), [n for _, n in present][::-1])
    ax.set_ylim(-0.6, len(present) - 0.4)
    ax.grid(axis="y", visible=False)
    ax.set_title(TASKS[task], fontsize=11)
    ax.set_xlabel("class-insensitive homophily (● all links, ○ no same-patient links)")
axes[0].legend(loc="lower right")
fig.tight_layout()
nf.save(fig, figures_dir / "11_robustness_homophily.png")
plt.show()

In [ ]:
out = write_aggregate(rows, results_dir / "robustness_checks.csv")
saved = pd.read_csv(out)
print(f"wrote {len(saved)} rows -> {out.relative_to(_repo_root)}")
saved.groupby(["task", "graph_variant"])["metric"].count()

## Findings

**Every Project 1 conclusion survives all three checks.** The size of the Task B leakage effect depends on how the graph is built, and the report should say so.

**1. Coding era is a strong structural split, but it explains almost none of the homophily.**

| Primary (raw-code) graphs | Long stay, safe | Hypertension, safe |
|---|---|---|
| Links joining an ICD-9-only and an ICD-10-only node | 3 of 18,698 (0.02%) | 6 of 10,535 (0.06%) |
| Era assortativity (1 = only ever within era) | 0.74 | 0.77 |
| Homophily from era alone (outcomes shuffled within era) | 0.004 | 0.019 |
| Actual homophily | **0.209** | **0.282** |

Era alone produces at most 2% (long stay) and 7% (hypertension) of the observed homophily. Outcome rates do differ by era for hypertension: 17.8% of ICD-9-era discharges against 8.3% of ICD-10-era ones. But nearly all the outcome signal lives *within* eras.

**2. SNOMED harmonization opens the era wall, and the conclusions hold.** Cross-era links rise from about 0.02–0.06% to 3–4% of links. Era assortativity only falls to 0.67–0.69: patients from different eras still differ in more than their codes. With the same pipeline on harmonized codes:

| | Long stay: raw → SNOMED | Hypertension: raw → SNOMED |
|---|---|---|
| Homophily, safe / leaky | 0.209 / 0.201 → 0.212 / 0.205 | 0.282 / 0.332 → 0.292 / 0.324 |
| Without same-patient links, safe / leaky | 0.237 / 0.242 → 0.241 / 0.234 | 0.136 / 0.193 → 0.147 / 0.180 |
| Outcome explained by community (η²), safe / leaky | 0.121 / 0.147 → 0.121 / 0.126 | 0.076 / 0.123 → 0.076 / 0.088 |
| Strongest single-code gap, leaky | 0.16 → 0.27 | 0.62 → **0.84** |

Long stay is unchanged: still no leakage effect. Hypertension's leaky graph still beats safe, but by less: +0.032 instead of +0.050 homophily. **The answer key gets *stronger* under SNOMED**, since ICD-9 401.9 and ICD-10 I10 merge into one concept present in 84 points more positives than negatives. **Yet its effect on the graph gets *weaker*.** A merged concept that common is heavily down-weighted by TF-IDF. This is the clearest evidence yet that how much leakage reaches a graph depends on how the graph is built.

**3. On the clean Hypertension cohort, the leakage effect is larger.** Applying EHRSHOT's own definition (SNOMED 59621000 and its 16-concept subtree), **55 of 515 positives (10.7%) and 84 of 3,245 negatives (2.6%) already have an essential-hypertension code before the prediction time**, a median of 14 days earlier, mostly within the same stay. Without those 139 discharges (3,621 left, prevalence 0.127):

| Hypertension | Official cohort | Clean cohort |
|---|---|---|
| Homophily, safe / leaky / null | 0.282 / 0.332 / 0.004 | **0.258 / 0.332** / 0.004 |
| Without same-patient links | 0.136 / 0.193 (+42%) | **0.108 / 0.188 (+74%)** |
| η², safe / leaky | 0.076 / 0.123 | 0.065 / 0.125 |

The prior-coded patients were inflating the *safe* graph's alignment, which is legitimate information but not a "new" diagnosis. Removing them widens the safe-leaky gap from +0.050 to +0.074.

### What the report can now say

1. **Honest patient graphs are organized by outcome, far beyond chance.** That holds in every version: raw codes, SNOMED, clean cohort, with or without same-patient links. Null homophily never exceeds 0.02.
2. **Leakage is inert for long stay and real for hypertension.** That also holds in every version. Hypertension's safe-leaky homophily gap ranges from +0.032 (SNOMED) to +0.074 (clean cohort) depending on construction.
3. **Three construction choices measurably change how much leakage a graph carries:** code harmonization, same-patient links, and cohort definition. That is a concrete argument for the Project 2 graph-variant sweep, with SNOMED harmonization as a core arm.

### Caveats

- The clean cohort departs from Stanford's official labels. It is a sensitivity analysis, not a replacement, and keeping the official cohort primary keeps our numbers comparable to the benchmark.
- Why EHRSHOT's exclusion missed these patients is not established. The shipped vocabulary does map I10 and 401.9 into the essential-hypertension subtree. It may be how the labeler times diagnoses within a stay. Worth a careful question to Stanford's team, not a claim.

### Done when
- [x] #35: coding era quantified, SNOMED-harmonized patient graphs analyzed for both tasks, clean Hypertension cohort analyzed
- [x] Aggregate output only: `results/aggregate/robustness_checks.csv` and `11_robustness_homophily.png`